<a href="https://colab.research.google.com/github/AlishbaMalik687-svg/AI-ML_internship_Adv_Task_2/blob/master/Adv_task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Enable GPU

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

 2. Install libraries

In [2]:

!pip install transformers datasets gradio -q

# 3. Import libraries

from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score




 3. Loading Dataset


In [3]:

dataset = load_dataset("HuyAugie/Smaller_AG_News_Dataset")

train_dataset = dataset["train"]
test_dataset = dataset["test"]



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ag_news_train_balanced.parquet:   0%|          | 0.00/651k [00:00<?, ?B/s]

ag_news_train_imbalanced.parquet:   0%|          | 0.00/313k [00:00<?, ?B/s]

ag_news_test_small.parquet:   0%|          | 0.00/650k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5900 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4000 [00:00<?, ? examples/s]

4. Load Tokenizer

In [4]:

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")   #Yahan hum bert-base-uncased ka tokenizer load kar rahe hain.Text ko numbers (tokens) me convert karna.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

 5. Tokenization Function

In [5]:
 # Yahan hum dataset ke har news text ko tokenizer se process kar rahe hain.
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,     # truncation=True → agar text lamba ho to cut ho jaye
        padding="max_length",     #padding="max_length" → sab inputs equal length ke ho jayein
        max_length=128
    )


# Apply tokenization
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)




Map:   0%|          | 0/5900 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

6. Load BERT Model

In [8]:
#Yahan hum BERT model load kar rahe hain jo already pretrained hai.
model = BertForSequenceClassification.from_pretrained(
   "bert-base-uncased",
    num_labels=4      #Matlab model 4 categories predict karega.
)

 #Hum isko fine-tune kar rahe hain news classification ke liye



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 7. Training Arguments

In [9]:
 #Yahan hum training settings define kar rahe hain.
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,        #model itni fast learn kare
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,      #Matlab model dataset 3 times train karega.
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
# is function se accuracy or fi scor pta chaly ga model ka

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')

    return {
        "accuracy": acc,
        "f1_score": f1
    }

 8. Trainer

In [11]:
#Trainer Hugging Face ka tool hai jo: training handle karta hai, evaluation karta hai, logging karta hai
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
 #yahan hum: training dataset, testing dataset, model, sab connect kar rahe hain.

9. Train Model

In [12]:
 #Ab model training start karta hai.
trainer.train()

Step,Training Loss
500,0.434440
1000,0.237531


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1476, training_loss=0.27089928934567664, metrics={'train_runtime': 396.233, 'train_samples_per_second': 29.78, 'train_steps_per_second': 3.725, 'total_flos': 776191551283200.0, 'train_loss': 0.27089928934567664, 'epoch': 2.0})

10.send model to device to avoid GPU and CPU mismatch error

In [13]:
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

11. Evaluate Model

In [14]:
#yahan model test dataset par check hota hai.
result = trainer.evaluate()
print("Evaluation Result:", result)

Evaluation Result: {'eval_loss': 0.3591669797897339, 'eval_accuracy': 0.9145, 'eval_f1_score': 0.9145457493828382, 'eval_runtime': 31.9654, 'eval_samples_per_second': 125.135, 'eval_steps_per_second': 15.642, 'epoch': 2.0}


12. Gradio Interface

In [15]:
#Is se simple web app ban jata hai jahan user news headline enter karta hai aur model topic predict karta hai.
import gradio as gr

labels = ["World", "Sports", "Business", "Sci/Tech"]

def predict(text):
    model.eval() # Set model to evaluation mode

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move tokenized inputs to the device
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = outputs.logits.argmax().item()

    return labels[pred]


demo = gr.Interface(
    fn=predict,
    inputs="text",
    outputs="text",
    title="News Topic Classifier"
)

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e7854d362c64aec3a6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e7854d362c64aec3a6.gradio.live
